In [1]:
import os
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from dotenv import load_dotenv
from sklearn.preprocessing import StandardScaler

In [2]:
#load dataset credentials from  .env
load_dotenv()

#connect to neon
engine = create_engine(os.getenv("DATABASE_URL"))
print("Connected to Neon")


Connected to Neon


In [4]:
#pulling the messy data
df = pd.read_sql("select * from customers",engine)
print(len(df))
print(len(df.columns))

df.head()

10000
12


,CustomerID,Name,Gender,Age,City,Signup_Date,Last_purchase_date,purchase_amount,feedback_score,email,Phone_number,Country
0,NaN,Ankit,F,36.0,Mumbai,31/12/2024,13/08/2025,-999.0,2.0,NaN,0,Canada
1,C2,Ravi,Female,NaN,Kolkata,NaN,NaN,NaN,-1.0,user1mail.com,abc123,Canada
2,3,Ravi,female,66.0,Ahmedabad,NaN,20/06/2023,NaN,10.0,user2@mail.com,abc123,India
3,C4,Ankit,male,44.0,Kolkata,NaN,13/09/2025,NaN,NaN,NaN,9316267914,USA
4,5,Rahul,Male,200.0,Ahmedabad,09/07/2025,NaN,-999.0,NaN,user4mail.com,9234603292,USA


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   CustomerID          7621 non-null   str    
 1   Name                8342 non-null   str    
 2   Gender              7545 non-null   str    
 3   Age                 7424 non-null   float64
 4   City                8870 non-null   str    
 5   Signup_Date         7471 non-null   str    
 6   Last_purchase_date  6714 non-null   str    
 7   purchase_amount     4967 non-null   float64
 8   feedback_score      7538 non-null   float64
 9   email               4934 non-null   str    
 10  Phone_number        7556 non-null   str    
 11  Country             8296 non-null   str    
dtypes: float64(3), str(9)
memory usage: 1.4 MB


In [6]:
df.describe()

,Age,purchase_amount,feedback_score
count,7424.000000,4967.000000,7538.000000
mean,78.904364,713.654008,4.008092
std,87.817619,2002.187228,4.619336
min,-5.000000,-999.000000,-1.000000
25%,-5.000000,-999.000000,-1.000000
50%,43.000000,-999.000000,3.000000
75%,200.000000,2437.600000,10.000000
max,200.000000,4998.670000,10.000000


In [7]:
# Check how many NULLs in each column
df.isnull().sum()


CustomerID            2379
Name                  1658
Gender                2455
Age                   2576
City                  1130
Signup_Date           2529
Last_purchase_date    3286
purchase_amount       5033
feedback_score        2462
email                 5066
Phone_number          2444
Country               1704
dtype: int64

In [8]:
# we fix the negative purchases
# replace -999 with 0
df['purchase_amount'] = df['purchase_amount'].replace(-999,0)
# we will check
print(df['purchase_amount'].min())

0.0


In [9]:
# filling the null values 
# for age we use median
df['Age'] = df['Age'].fillna(df['Age'].median())

# Fill NULL Emails with 'unknown@missing.com'
df['email'] = df['email'].fillna('unknown@missing.com')

# Fill NULL Phones with '0000000000'
df['Phone_number'] = df['Phone_number'].fillna('0000000000')

# Check if NULLs are gone
print(df.isnull().sum())


CustomerID            2379
Name                  1658
Gender                2455
Age                      0
City                  1130
Signup_Date           2529
Last_purchase_date    3286
purchase_amount       5033
feedback_score        2462
email                    0
Phone_number             0
Country               1704
dtype: int64


In [22]:
# encode categorical columns
#encode gender
df['Gender_encoded'] = df['Gender'].map({'M':1, 'F':0, 'Unknown': 2})

# encode country (asign each number to country)
df['Country_encoded'] = df['Country'].astype('category').cat.codes
# Check the new columns
df[['Gender', 'Gender_encoded', 'Country', 'Country_encoded']].head()


,Gender,Gender_encoded,Country,Country_encoded
0,F,0.0,Canada,1
1,Female,NaN,Canada,1
2,female,NaN,India,2
3,male,NaN,USA,4
4,Male,NaN,USA,4


In [23]:
# Drop columns we don't need for ML in to new df_ml
df_ml = df.drop(columns=['Name', 'City', 'Gender', 'Country'])

# See the final cleaned data
df_ml.head()

,CustomerID,Age,Signup_Date,Last_purchase_date,purchase_amount,feedback_score,email,Phone_number,Gender_encoded,Country_encoded
0,NaN,36.0,31/12/2024,13/08/2025,0.0,2.0,unknown@missing.com,0,0.0,1
1,C2,43.0,NaN,NaN,NaN,-1.0,user1mail.com,abc123,NaN,1
2,3,66.0,NaN,20/06/2023,NaN,10.0,user2@mail.com,abc123,NaN,2
3,C4,44.0,NaN,13/09/2025,NaN,NaN,unknown@missing.com,9316267914,NaN,4
4,5,200.0,09/07/2025,NaN,0.0,NaN,user4mail.com,9234603292,NaN,4


In [24]:
# Define which columns to scale
numeric_cols = ['Age', 'purchase_amount', 'feedback_score']

# Fit the scaler and transform
scaler = StandardScaler()
df_ml[['Age_scaled', 'Purchase_scaled', 'Feedback_scaled']] = scaler.fit_transform(df_ml[numeric_cols])

# Drop the original unscaled columns
df_ml = df_ml.drop(columns=numeric_cols)

# Check the new scaled data
df_ml.head()

,CustomerID,Signup_Date,Last_purchase_date,email,Phone_number,Gender_encoded,Country_encoded,Age_scaled,Purchase_scaled,Feedback_scaled
0,NaN,31/12/2024,13/08/2025,unknown@missing.com,0,0.0,1,-0.435539,-0.764246,-0.434743
1,C2,NaN,NaN,user1mail.com,abc123,NaN,1,-0.344951,NaN,-1.084230
2,3,NaN,20/06/2023,user2@mail.com,abc123,NaN,2,-0.047305,NaN,1.297222
3,C4,NaN,13/09/2025,unknown@missing.com,9316267914,NaN,4,-0.332010,NaN,NaN
4,5,09/07/2025,NaN,user4mail.com,9234603292,NaN,4,1.686808,-0.764246,NaN


In [25]:
# save the cleaned data
# Create a 'data/' folder if it doesn't exist
os.makedirs('data', exist_ok=True)

# Save to CSV
df_ml.to_csv('data/cleaned_customers.csv', index=False)
print(" Cleaned data saved to 'data/cleaned_customers.csv'")

 Cleaned data saved to 'data/cleaned_customers.csv'


In [26]:
# at last final view
print(f"Total Rows: {len(df_ml)}")
print(f"Total Columns: {len(df_ml.columns)}")
print(f"Columns: {list(df_ml.columns)}")
print("\nSample of final data:")
df_ml.head()

Total Rows: 10000
Total Columns: 10
Columns: ['CustomerID', 'Signup_Date', 'Last_purchase_date', 'email', 'Phone_number', 'Gender_encoded', 'Country_encoded', 'Age_scaled', 'Purchase_scaled', 'Feedback_scaled']

Sample of final data:


,CustomerID,Signup_Date,Last_purchase_date,email,Phone_number,Gender_encoded,Country_encoded,Age_scaled,Purchase_scaled,Feedback_scaled
0,NaN,31/12/2024,13/08/2025,unknown@missing.com,0,0.0,1,-0.435539,-0.764246,-0.434743
1,C2,NaN,NaN,user1mail.com,abc123,NaN,1,-0.344951,NaN,-1.084230
2,3,NaN,20/06/2023,user2@mail.com,abc123,NaN,2,-0.047305,NaN,1.297222
3,C4,NaN,13/09/2025,unknown@missing.com,9316267914,NaN,4,-0.332010,NaN,NaN
4,5,09/07/2025,NaN,user4mail.com,9234603292,NaN,4,1.686808,-0.764246,NaN


In [21]:
# we have filled nulls as per the need of columns in the dataset